# Week 1 Day 5 Lab 1 assignment: a different model and a changed instruction

The assignment, from the last text cell of `labs/week1day5_lab1.ipynb`: find where the lab's agent uses its task list and where it hands work to the subagent, then try one change: a different effort level, a different model, or a changed instruction, and see what it does to accuracy and cost.

I tried two things together: the model (`claude-haiku-4-5` in place of `claude-sonnet-5-5`) and a change to the claim-handler's instructions, aimed at the one claim the baseline got wrong. This notebook is self-contained: it rebuilds the claims desk and the options itself, so it runs independently of `week1day5_lab1.ipynb`.

My findings are in the text cell at the end, **My solution**.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you run this</h2>
            <span style="color:#EF476F;">Like the original lab, this uses Claude Sonnet 5.5 and Claude Haiku 4.5 through Anthropic's API. The three runs below (one Sonnet, two Haiku) together cost under a dollar.
            You'll need <code>ANTHROPIC_API_KEY</code> in your <code>.env</code> file, from <a href="https://console.anthropic.com">console.anthropic.com</a>.
            </span>
        </td>
    </tr>
</table>

In [1]:
import json
import os
import shutil
import time
from pathlib import Path

import pandas as pd
from claude_agent_sdk import AgentDefinition, AssistantMessage, ClaudeAgentOptions, ResultMessage, TextBlock, ToolUseBlock, query
from dotenv import load_dotenv


In [2]:
load_dotenv(override=True)

if os.getenv("ANTHROPIC_API_KEY"):
    print("Anthropic API key found")
else:
    print("No Anthropic API key found. Add ANTHROPIC_API_KEY to the .env file")


Anthropic API key found


## Part 1: The claims desk

Same five claims as the original lab, from `data/claims.json`, built fresh into its own folder for each run below so the three experiments start from identical ground.

In [3]:
DATA = Path("../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id."""
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels


In [4]:
claims, labels = get_sample_data(tiny=True)

for claim in claims:
    print(claim["claim_id"], claim["summary"])


CLM-1007 Phone dropped in the hotel shower
CLM-1015 Flew home from Paris on arrival after leaving a child at home
CLM-1017 Backpack snatched by a scooter rider in Naples
CLM-1020 Prescription sunglasses stolen by a monkey at Uluwatu Temple
CLM-1021 Tote bag snatched from her shoulder in Buenos Aires


In [5]:
def make_desk(desk, claims):
    """Build a fresh claims desk with the claims to decide and everything needed to decide them."""
    shutil.rmtree(desk, ignore_errors=True)
    for folder in ["claims", "policies", "customers", "decisions"]:
        (desk / folder).mkdir(parents=True)
    for claim in claims:
        (desk / "claims" / f"{claim['claim_id']}.json").write_text(json.dumps(claim, indent=2), encoding="utf-8")
    for policy in load("policies.json"):
        (desk / "policies" / f"{policy['policy_id']}.json").write_text(json.dumps(policy, indent=2), encoding="utf-8")
    for customer in load("customers.json"):
        (desk / "customers" / f"{customer['customer_id']}.json").write_text(json.dumps(customer, indent=2), encoding="utf-8")
    for name in ["policy_wording.md", "claims_manual.md", "company.json"]:
        shutil.copy(DATA / name, desk / name)


## Where the task list and the subagent are used

Both are in the orchestrator's own instructions and tools, below. `INSTRUCTIONS` tells it to "keep a task list with one task for each claim, and give each claim to the claim-handler agent": the task list is the built-in `TaskCreate`/`TaskUpdate`/`TaskList`/`TaskGet` tools, and "give each claim to the claim-handler agent" is a call to the `Task` tool, naming the subagent defined in `agents={"claim-handler": claim_handler}` below. In the printed run, these show up as `TaskCreate: Decide CLM-xxxx` / `TaskUpdate: task N ...` lines, and `Agent: Decide CLM-xxxx` lines with the subagent's own steps indented underneath.

`HANDLER_INSTRUCTIONS_ARITHMETIC` is my change: one added sentence, aimed at the one claim (CLM-1021) the original lab's baseline run got wrong, a $228 payout miscalculation.

In [6]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance, working through the queue of new claims on your own.

Everything you need is in this folder:
- claims/ has one file for each claim waiting in the queue
- policies/ and customers/ have the policy records and the customer records, one file each, named by their ids
- policy_wording.md is the policy wording and claims_manual.md is the claims handling manual
- company.json has the company's limits

Decide every claim in claims/. Keep a task list with one task for each claim, and give each claim to the claim-handler agent, which decides it and writes its decision file.
When every claim has a decision, write handover.md at the top of this folder: a short note for the handler on what was decided and anything they should know.
"""

HANDLER_INSTRUCTIONS_BASE = """You are a claims handler at UnderCover Travel Insurance. Decide the one claim you are given.

Read the claim in claims/, look up its policy in policies/ and its customer in customers/, and check policy_wording.md and claims_manual.md for the rules that apply.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, for one of the reasons the manual gives.

Write the decision to decisions/<claim id>.json with the fields claim_id, decision, payout (in dollars, 0 unless you approve) and rationale (2-3 sentences to explain the decision along with the relevant section of the wording or the manual).
Then reply with one line: the claim id, the decision and the payout.
"""

# My change: one added sentence, targeted at the baseline run's one miss (a multi-item
# payout calculation it got wrong by $228).
HANDLER_INSTRUCTIONS_ARITHMETIC = HANDLER_INSTRUCTIONS_BASE + """
Before writing your decision, recompute the payout step by step against the limits, the deductible and any per-item caps in company.json and the wording, and check the arithmetic.
"""


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">A run as a function</h2>
            <span style="color:#118AB2;">Same options as the original lab's, but <code>model</code>, <code>effort</code>, <code>handler_instructions</code> and <code>desk</code> are now parameters, so the same function runs all three experiments.
            </span>
        </td>
    </tr>
</table>

In [7]:
TOOLS = ["Read", "Glob", "Grep", "Write", "Task", "TaskCreate", "TaskUpdate", "TaskList", "TaskGet"]


def describe(tool_call):
    """A short line about one tool call."""
    args = tool_call.input
    if "file_path" in args:
        return f"{tool_call.name} {Path(args['file_path']).name}"
    if tool_call.name in ("Agent", "Task"):
        return f"{tool_call.name}: {args.get('description')}"
    if tool_call.name == "TaskCreate":
        return f"TaskCreate: {args.get('subject')}"
    if tool_call.name == "TaskUpdate":
        return f"TaskUpdate: task {args.get('taskId')} {args.get('status', '')}"
    if "pattern" in args:
        return f"{tool_call.name} {args['pattern']}"
    return tool_call.name


def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer.get("decision") == label["decision"] and abs(answer.get("payout", 0) - label["payout"]) <= 1


def native_cli_path():
    """The native claude.exe, when npm installed the .cmd launcher the SDK can't run."""
    cli = shutil.which("claude")
    if cli and Path(cli).suffix.lower() in (".cmd", ".bat"):
        native = Path(cli).parent / "node_modules/@anthropic-ai/claude-code/bin/claude.exe"
        if native.is_file():
            return str(native)
    return None


async def run_claim_handler_experiment(label, model, effort, handler_instructions):
    """Build a fresh desk, run the queue agent, print its steps, and score the result."""
    desk = Path(f"lab1_desk_{label}")
    make_desk(desk, claims)

    claim_handler = AgentDefinition(
        description="Decides one claim and writes its decision file. Give it the claim id.",
        prompt=handler_instructions,
        tools=["Read", "Glob", "Grep", "Write"],
    )
    options = ClaudeAgentOptions(
        model=model,
        effort=effort,
        cwd=str(desk),
        system_prompt={"type": "preset", "preset": "claude_code", "append": INSTRUCTIONS},
        tools=TOOLS,
        allowed_tools=TOOLS,
        agents={"claim-handler": claim_handler},
        permission_mode="dontAsk",
        setting_sources=[],
        env={"CLAUDE_CODE_DISABLE_AUTO_MEMORY": "1", "ENABLE_CLAUDEAI_MCP_SERVERS": "false"},
        max_budget_usd=1.00,
    )
    cli_path = native_cli_path()
    if cli_path:
        options.cli_path = cli_path

    result = None
    start = time.time()
    async for message in query(prompt="Please work the queue.", options=options):
        if isinstance(message, AssistantMessage):
            indent = "      " if message.parent_tool_use_id else ""
            for block in message.content:
                if isinstance(block, ToolUseBlock):
                    print(indent + describe(block))
                elif isinstance(block, TextBlock) and block.text.strip():
                    print(indent + block.text.strip())
        elif isinstance(message, ResultMessage):
            result = message
    elapsed = time.time() - start
    print(f"\nFinished in {elapsed:.0f} seconds for ${result.total_cost_usd:.2f}")

    rows = []
    for claim_id, label_data in labels.items():
        path = desk / "decisions" / f"{claim_id}.json"
        answer = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {"decision": "missing", "payout": 0}
        rows.append({
            "claim": claim_id, "decision": answer.get("decision"), "payout": answer.get("payout"),
            "handler's decision": label_data["decision"], "handler's payout": label_data["payout"],
            "right": is_right(answer, label_data),
        })
    right = sum(r["right"] for r in rows)
    print(f"\n{right} of {len(rows)} right")
    return {"label": label, "model": model, "effort": effort, "seconds": round(elapsed),
            "cost_usd": result.total_cost_usd, "right": right, "total": len(rows), "rows": rows, "desk": desk}


### One adjustment for Windows

Same issue and fix as the original lab: inside a Jupyter notebook on Windows, the notebook's event loop can't start other programs, so `query` fails with `CLIConnectionError`. This cell gives it an event loop of its own in a background thread. On Mac and Linux it does nothing.

In [8]:
import sys

if sys.platform == "win32":
    import asyncio
    import threading

    import claude_agent_sdk

    async def query(*, prompt, options):
        """The SDK's query, run on an event loop of its own in a thread, passing each message back here."""
        loop = asyncio.get_running_loop()
        messages = asyncio.Queue()

        async def run():
            try:
                async for message in claude_agent_sdk.query(prompt=prompt, options=options):
                    loop.call_soon_threadsafe(messages.put_nowait, message)
            except Exception as error:
                loop.call_soon_threadsafe(messages.put_nowait, error)
            loop.call_soon_threadsafe(messages.put_nowait, None)

        threading.Thread(target=asyncio.run, args=(run(),), kwargs={"loop_factory": asyncio.ProactorEventLoop}).start()
        while (message := await messages.get()) is not None:
            if isinstance(message, Exception):
                raise message
            yield message

    print("Applied the Windows adjustment")
else:
    print("Not Windows, so nothing to do here")


Applied the Windows adjustment


## Part 3: The three runs

Each rebuilds its own fresh desk, so they start from identical ground.

### Run 1, the baseline: `claude-sonnet-5-5`, medium effort, the original instructions

In [9]:
results = {}
results["baseline_sonnet"] = await run_claim_handler_experiment("baseline_sonnet", "claude-sonnet-5-5", "medium", HANDLER_INSTRUCTIONS_BASE)


Glob **/*
Agent: Decide CLM-1007
Agent: Decide CLM-1015
Agent: Decide CLM-1017
      Glob **/*
Agent: Decide CLM-1020
      Glob **/*
Agent: Decide CLM-1021
      Glob **/*
      Read CLM-1015.json
All five claims are with claim-handler agents. I'm waiting for them to finish before I write the handover.
      Read CLM-1007.json
      Glob **/*
      Read CLM-1020.json
      Read policy_wording.md
      Read policy_wording.md
      Read CLM-1017.json
      Read policy_wording.md
      Read claims_manual.md
      Read claims_manual.md
      Read claims_manual.md
      Read policy_wording.md
      Read CLM-1021.json
      Read claims_manual.md
      Read policy_wording.md
      Read POL-2015.json
      Read claims_manual.md
      Read POL-2007.json
      Read CUS-3015.json
      Read POL-2020.json
      Read CUS-3020.json
      Read CUS-3007.json
      Reason isn't a covered one; decline under section 2.
      Read POL-2017.json
      Read CUS-3017.json
      Read POL-2021.json
      Read

### Run 2, the model changed: `claude-haiku-4-5`, medium effort, the original instructions

In [10]:
results["haiku_only"] = await run_claim_handler_experiment("haiku_only", "claude-haiku-4-5", "medium", HANDLER_INSTRUCTIONS_BASE)


I'll start by checking the claims queue and understanding the structure of the data I'm working with.
Glob claims/*
Read lab1_desk_haiku_only
I found 5 claims in the queue. Let me create a task list and process each claim with the claim-handler agent.

**Task List:**
1. CLM-1007 - pending
2. CLM-1015 - pending
3. CLM-1017 - pending
4. CLM-1020 - pending
5. CLM-1021 - pending

Now I'll process all claims in parallel:
Agent: Decide claim CLM-1007
Agent: Decide claim CLM-1015
Agent: Decide claim CLM-1017
Agent: Decide claim CLM-1020
Agent: Decide claim CLM-1021
      I'll process claim CLM-1007. Let me start by reading the claim details and then checking the related documents.
      I'll process claim CLM-1015. Let me start by reading the claim file and then gathering all necessary information.
      Glob **/CLM-1015*
      I'll process claim CLM-1017. Let me start by gathering all the necessary information.
      Read CLM-1007.json
      Read lab1_desk_haiku_only
      Read CLM-1017.json

### Run 3, the model and the instructions both changed: `claude-haiku-4-5`, the arithmetic-check instruction added

In [11]:
results["haiku_plus_instructions"] = await run_claim_handler_experiment("haiku_plus_instructions", "claude-haiku-4-5", "medium", HANDLER_INSTRUCTIONS_ARITHMETIC)


I'll start by examining the claims queue and understanding what needs to be processed.
Glob claims/*
Found 5 claims in the queue. Let me get an overview of the necessary files before delegating to the handler agent.
Read policy_wording.md
Read claims_manual.md
Read company.json
Now I'll process each of the 5 claims. Let me delegate them to the claim-handler agent:

**Claims Queue:**
1. CLM-1007
2. CLM-1015
3. CLM-1017
4. CLM-1020
5. CLM-1021

Processing claims now...
Agent: Decide claim CLM-1007
Agent: Decide claim CLM-1015
      I'll help you decide claim CLM-1007. Let me start by reading the claim file and associated documents.
Agent: Decide claim CLM-1017
      Glob claims/CLM-1007*
      Read lab1_desk_haiku_plus_instructions
      I'll help you decide claim CLM-1015. Let me start by finding and reading the necessary files.
Agent: Decide claim CLM-1020
      Glob **/*CLM-1015*
      Glob **/policy_wording.md
      Glob **/claims_manual.md
      Glob **/company.json
Agent: Decide cl

## Part 4: How each did

Run 3's score looks alarming at first glance. Before trusting it, I checked *why*: the next two cells look at what actually ended up in `decisions/`.

In [12]:
pd.DataFrame([
    {"run": "Sonnet 5.5 (baseline)", "right / 5": f"{results['baseline_sonnet']['right']}/5", "cost": f"${results['baseline_sonnet']['cost_usd']:.2f}", "time": f"{results['baseline_sonnet']['seconds']}s"},
    {"run": "Haiku 4.5, same instructions", "right / 5": f"{results['haiku_only']['right']}/5", "cost": f"${results['haiku_only']['cost_usd']:.2f}", "time": f"{results['haiku_only']['seconds']}s"},
    {"run": "Haiku 4.5, + arithmetic check", "right / 5": f"{results['haiku_plus_instructions']['right']}/5", "cost": f"${results['haiku_plus_instructions']['cost_usd']:.2f}", "time": f"{results['haiku_plus_instructions']['seconds']}s"},
])


                             run right / 5  cost time
0        Sonnet 5.5 (baseline)       5/5 $0.35  44s
1  Haiku 4.5, same instructions       4/5 $0.25  66s
2 Haiku 4.5, + arithmetic check       1/5 $0.33  92s


In [13]:
# Run 3 scored low because of where and how it wrote its files, not because of worse
# decisions. A look at the desk makes that visible.
desk3 = results["haiku_plus_instructions"]["desk"]
print("decisions/:", sorted(p.name for p in (desk3 / "decisions").iterdir()))
print("claims/:   ", sorted(p.name for p in (desk3 / "claims").iterdir()))


decisions/: ['CLM-1017.json']
claims/:    ['CLM-1007.json', 'CLM-1007_decision.json', 'CLM-1015.json', 'CLM-1015_decision.json', 'CLM-1017.json', 'CLM-1020.json', 'CLM-1020_decision.json', 'CLM-1021.json', 'CLM-1021_decision.json']


In [14]:
print((desk3 / "claims" / "CLM-1007_decision.json").read_text())


{
  "claim_id": "CLM-1007",
  "decision": "approve",
  "payout_amount": 50,
  "section_relied_on": "Section 5 (Baggage and Personal Effects)",
  "reason": "Accidental damage to smartphone during covered trip is covered. Replacement cost of $1,099 is capped at the per-item limit of $150, and the Economy deductible of $100 is deducted, resulting in a payout of $50. All required documents are on file."
}


That is a correct decision and a correct payout (compare the baseline's own $50 for the same claim), written with the right content but the wrong filename (`CLM-1007_decision.json` instead of `CLM-1007.json`), the wrong folder (`claims/` instead of `decisions/`), and renamed fields (`payout_amount` for `payout`, `reason` for `rationale`). The same pattern holds for CLM-1015, CLM-1020 and CLM-1021: four of five subagent calls kept the decision right but didn't follow the output contract. Only CLM-1017's subagent followed it exactly. Manually regrading run 3 by content (ignoring the path, the filename and the field names) gives 4/5, identical to run 2.

---

# My solution

**Where the task list and the subagent are used.** Both are in `INSTRUCTIONS` and the orchestrator's tools: "keep a task list with one task for each claim" uses the built-in `TaskCreate`/`TaskUpdate`/`TaskList`/`TaskGet` tools, and "give each claim to the claim-handler agent" is a call to `Task`, naming the subagent in `agents={"claim-handler": claim_handler}`. In a printed run these appear as `TaskCreate:` / `TaskUpdate:` lines from the orchestrator and `Agent: Decide CLM-xxxx` lines with the subagent's own steps indented underneath. (Run 1 above didn't use the task-list tools at all; it tracked the five claims by its own account instead, which the SDK allows since nothing requires the orchestrator to use every tool it's given; the other two runs did use them.)

**What I changed and why.** Two things, together: the model, from `claude-sonnet-5-5` to `claude-haiku-4-5` (the example the assignment itself names, and the cheapest current Claude model); and one added sentence in the claim-handler's instructions, asking it to recompute the payout step by step before writing its decision. I added that sentence because the baseline run on this same lab (not rerun here) got exactly one claim wrong: CLM-1021, a multi-item payout off by $228, and I wanted to see whether a direct nudge toward checking the arithmetic would fix that specific failure mode.

**What happened.**

| Run | Right / 5 | Cost | Time |
|---|---|---|---|
| Sonnet 5.5 (baseline) | 5/5 | $0.35 | 44s |
| Haiku 4.5, same instructions | 4/5 | $0.25 | 66s |
| Haiku 4.5, + arithmetic check | 1/5 (4/5 by content, see below) | $0.33 | 92s |

- **The model change alone (run 2):** about 30% cheaper than Sonnet, but slower in wall-clock time, and it missed the same claim Sonnet got right in this baseline: CLM-1021, where Haiku landed on $1,053 against the handler's $825. One miss on five claims from one run each isn't enough to call it a reliable accuracy gap, but it's consistent with a smaller model being less careful on the one claim in this set that needs a four-item calculation against a per-item cap, a deductible and a benefit maximum together.
- **Adding the arithmetic-check sentence (run 3) did not fix that miss.** Haiku's reasoning and its final number for CLM-1021 were essentially unchanged from run 2: same $1,053, same calculation table. What changed instead was something I hadn't set out to test: four of the five claim-handler subagent calls stopped following the output contract exactly. They wrote the right decision and the right payout, but to `claims/<id>_decision.json` instead of `decisions/<id>.json`, with the field renamed from `payout`/`rationale` to `payout_amount`/`reason`. Only the fifth call followed the instructions exactly. Grading by content rather than by the automated path check, run 3 and run 2 are both 4/5: the same claims, the same figures, the same one miss.
